# 天气预测 - LSTM 模型

## 运行环境说明

**执行本 notebook 前必须先激活 venv：**
```powershell
.\venv\Scripts\activate
```

## 任务说明

- 数据集：`cleaned_weather.csv`，原始每 10 分钟一条，重采样为每 30 分钟一条（每天 48 条）
- 滑动窗口：每 5 天一组，前 4 天（4×48 = 192 步）作为输入，第 5 天第一个时刻的温度 T 作为预测目标
- 训练特征：T（温度）、rh（湿度）、p（气压）
- 模型：单层 LSTM + 全连接
- 训练：10 轮，每轮打印损失值（MSE / MAE）和准确率（基于绝对误差阈值）


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

print('numpy:', np.__version__)
print('torch:', torch.__version__)
print('from_numpy 可用:', torch.from_numpy(np.array([1,2,3])).tolist())

numpy: 1.26.4
torch: 2.0.1+cu118
from_numpy 可用: [1, 2, 3]


In [2]:
df = pd.read_csv('./cleaned_weather.csv')
# 时间解析
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
df.head()

,p,T,Tpot,Tdew,rh,VPmax,VPact,VPdef,sh,H2OC,rho,wv,max. wv,wd,rain,raining,SWDR,PAR,max. PAR,Tlog
date,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:10:00,1008.89,0.71,273.18,-1.33,86.1,6.43,5.54,0.89,3.42,5.49,1280.62,1.02,1.60,224.3,0.0,0.0,0.0,0.0,0.0,11.45
2020-01-01 00:20:00,1008.76,0.75,273.22,-1.44,85.2,6.45,5.49,0.95,3.39,5.45,1280.33,0.43,0.84,206.8,0.0,0.0,0.0,0.0,0.0,11.51
2020-01-01 00:30:00,1008.66,0.73,273.21,-1.48,85.1,6.44,5.48,0.96,3.39,5.43,1280.29,0.61,1.48,197.1,0.0,0.0,0.0,0.0,0.0,11.60
2020-01-01 00:40:00,1008.64,0.37,272.86,-1.64,86.3,6.27,5.41,0.86,3.35,5.37,1281.97,1.11,1.48,206.4,0.0,0.0,0.0,0.0,0.0,11.70
2020-01-01 00:50:00,1008.61,0.33,272.82,-1.50,87.4,6.26,5.47,0.79,3.38,5.42,1282.08,0.49,1.40,209.6,0.0,0.0,0.0,0.0,0.0,11.81


In [3]:
def load_and_process_data(csv_path):
    """
    读取气象csv、时间解析、重采样30min、清洗、标准化、构造样本
    返回：(X_train, y_train), (X_test, y_test), scaler_feat, scaler_label
    """
    df = pd.read_csv(csv_path)
    # 时间解析
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

    # 选择特征：T(温度)、rh(湿度)、p(气压)
    use_cols = ["T", "rh", "p"]
    df = df[use_cols].copy()

    # 重采样30分钟，取均值，填充少量缺失
    df = df.resample("30min").mean()
    df = df.dropna()

    # -------- 时序分割：前70%训练，后30%测试，不能打乱 --------
    total_len = len(df)
    train_size = int(total_len * 0.7)
    df_train_raw = df.iloc[:train_size]
    df_test_raw = df.iloc[train_size:]

    # -------- Z-score 标准化：仅 fit 训练集，防止泄露 --------
    scaler_feat = StandardScaler()
    train_np = scaler_feat.fit_transform(df_train_raw.values)
    test_np = scaler_feat.transform(df_test_raw.values)

    # 目标 T 是第 0 列，单独保存标签反归一化器（方便后面还原真实温度）
    scaler_label = StandardScaler()
    scaler_label.fit(df_train_raw[["T"]])

    # ----------------窗口参数----------------
    steps_per_day = 48          # 30min 一条，一天 48 步
    input_days = 4
    pred_delay_days = 5
    seq_len = input_days * steps_per_day          # 192，输入 4 天
    window_total = pred_delay_days * steps_per_day # 240，完整 5 天窗口

    def create_samples(data_np):
        samples = []
        labels = []
        max_idx = len(data_np) - window_total
        for i in range(max_idx):
            window = data_np[i:i+window_total, :]
            feature = window[:seq_len, :]   # 前4天：0~191
            label = window[seq_len, 0]      # 第5天第一个时刻，T(第0列)
            samples.append(feature)
            labels.append(label)
        return np.array(samples), np.array(labels)

    X_train, y_train = create_samples(train_np)
    X_test, y_test = create_samples(test_np)

    print(f"训练集样本数：{len(X_train)}，测试集样本数：{len(X_test)}")
    print(f"每个样本 shape：X={X_train.shape[1:]}，y=标量")
    return (X_train, y_train), (X_test, y_test), scaler_feat, scaler_label

In [4]:
class WeatherDataset(Dataset):
    def __init__(self, X, y):
        # 兼容 numpy 2.x 与 torch 旧版的写法：用 tensor() 代替 from_numpy()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [5]:
class LstmNet(nn.Module):
    def __init__(self, n_features, hidden_size=64):
        super(LstmNet, self).__init__()
        self.hidden_size = hidden_size
        # batch_first=True → 输入输出 shape [batch, seq_len, feature]
        self.rnn = nn.LSTM(input_size=n_features, hidden_size=hidden_size,
                           num_layers=1, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, inputs):
        # inputs: [batch, seq_len, n_feat]
        _, (hs, cs) = self.rnn(inputs)
        # hs shape: [num_layers, batch, hidden_size]
        x = F.dropout(hs[0])
        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x)
        x = self.fc2(x)
        return torch.squeeze(x)  # [batch,1] → [batch]

In [6]:
def calc_accuracy(pred, label, threshold=0.5):
    """
    回归任务的准确率：标准化空间内绝对误差小于阈值算正确
    threshold=0.5 约等于真实温度误差 ±2℃（取决于训练集 T 的标准差）
    """
    correct = (torch.abs(pred - label) < threshold).float()
    return correct.mean().item() * 100


def train_one_epoch(model, loader, optimizer, criterion, device, acc_threshold=0.5):
    model.train()
    total_loss = 0.0
    total_mae = 0.0
    total_acc = 0.0
    count = 0
    for feat, label in loader:
        feat = feat.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        pred = model(feat)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_mae += torch.mean(torch.abs(pred - label)).item()
        total_acc += calc_accuracy(pred, label, acc_threshold)
        count += 1

    avg_loss = total_loss / count
    avg_mae = total_mae / count
    avg_acc = total_acc / count
    return avg_loss, avg_mae, avg_acc

In [7]:
def test_model(model, loader, criterion, device, acc_threshold=0.5):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_acc = 0.0
    count = 0
    with torch.no_grad():
        for feat, label in loader:
            feat = feat.to(device)
            label = label.to(device)
            pred = model(feat)
            loss = criterion(pred, label)
            total_loss += loss.item()
            total_mae += torch.mean(torch.abs(pred - label)).item()
            total_acc += calc_accuracy(pred, label, acc_threshold)
            count += 1
    avg_loss = total_loss / count
    avg_mae = total_mae / count
    avg_acc = total_acc / count
    return avg_loss, avg_mae, avg_acc

In [10]:
if __name__ == "__main__":
    # 1. 读取处理数据
    (X_train, y_train), (X_test, y_test), scaler_feat, scaler_label = load_and_process_data("./cleaned_weather.csv")

    # 构建dataset dataloader，时序任务 shuffle=False！
    train_ds = WeatherDataset(X_train, y_train)
    test_ds = WeatherDataset(X_test, y_test)

    batch_size = 16
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # 设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备：{device}")
    n_feature = 3
    model = LstmNet(n_features=n_feature, hidden_size=64).to(device)

    # 损失函数、优化器
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    epoch_num = 50
    print(f"开始训练，共 {epoch_num} 轮\n")
    print(f"{'Epoch':>6} | {'train_loss':>10} | {'train_MAE':>10} | {'train_acc':>9} | {'test_loss':>10} | {'test_MAE':>10} | {'test_acc':>9}")
    print('-' * 90)
    for epoch in range(1, epoch_num + 1):
        train_loss, train_mae, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_mae, test_acc = test_model(model, test_loader, criterion, device)
        print(f"{epoch:>6d} | {train_loss:>10.4f} | {train_mae:>10.4f} | {train_acc:>8.2f}% | {test_loss:>10.4f} | {test_mae:>10.4f} | {test_acc:>8.2f}%")

    print('\n训练完成！')
    # 保存模型
    # torch.save(model.state_dict(), './weather_lstm.pth')
    # print('模型已保存到 ./weather_lstm.pth')

训练集样本数：12056，测试集样本数：5031
每个样本 shape：X=(192, 3)，y=标量
使用设备：cuda
开始训练，共 50 轮

 Epoch | train_loss |  train_MAE | train_acc |  test_loss |   test_MAE |  test_acc
------------------------------------------------------------------------------------------
     1 |     0.1272 |     0.2661 |    85.26% |     0.1270 |     0.2777 |    82.19%
     2 |     0.0656 |     0.1881 |    94.10% |     0.0916 |     0.2245 |    88.14%
     3 |     0.0609 |     0.1797 |    94.57% |     0.0843 |     0.2129 |    89.52%
     4 |     0.0556 |     0.1715 |    95.29% |     0.0721 |     0.1953 |    92.15%
     5 |     0.0546 |     0.1697 |    95.32% |     0.0636 |     0.1831 |    92.89%
     6 |     0.0516 |     0.1649 |    95.71% |     0.0630 |     0.1826 |    93.34%
     7 |     0.0528 |     0.1665 |    95.56% |     0.0573 |     0.1743 |    93.98%
     8 |     0.0531 |     0.1664 |    95.28% |     0.0665 |     0.1883 |    92.83%
     9 |     0.0522 |     0.1651 |    95.66% |     0.0548 |     0.1711 |    94.68%
    